# Hailo AI HAT+ — Hardware Inference Benchmark

This notebook benchmarks neural network inference on the Hailo HAILO8L NPU
against our existing CPU pipeline and documents the full model compilation path.

**Three-part structure:**

1. **Hardware throughput benchmark** — ResNet-50 pre-compiled HEF (no scheduler overhead).
   Establishes the raw NPU capability: what the chip can do at saturation.
2. **Production deployment benchmark** — Our trained EfficientNet-B0 HEF vs CPU baseline.
   Real wall-clock latency under ROUND_ROBIN scheduling. This is what matters for Avis.
3. **Compilation pipeline documentation** — How we got from `frozen_extractor.pt` to a
   running `.hef`. Reproducible steps with all decisions recorded.

**Hardware under test:**
- Raspberry Pi 5 (8GB), Debian Trixie, Python 3.13.5
- Hailo AI HAT+ (HAILO8L, firmware 4.23.0, PCIe 0001:01:00.0)
- HailoRT 4.23.0 (system-wide, `/usr/lib/python3/dist-packages/hailort`)

**Key finding:**
- Raw NPU throughput: **0.21ms** per inference (ResNet-50, no scheduler)
- Production EfficientNet-B0 on HAILO8L: **13.13ms** mean (ROUND_ROBIN scheduler)
- CPU EfficientNet-B0 baseline: **85.43ms** mean
- **6.5× speedup** on our actual production model

**Run this notebook on the Pi** (not the laptop — requires Hailo hardware):
```bash
ssh birdfeeder01@<pi-ip>
cd /mnt/data/avis-birdfeeder
source /mnt/data/avis-venv/bin/activate
jupyter notebook --no-browser --port=8888
```

**Outputs:**
- `notebooks/results/hailo_benchmark_comparison.png` — bar chart comparing all paths
- `notebooks/results/hailo_latency_distribution.png` — latency distribution across 50 runs
- `notebooks/results/experiments.csv` — new rows appended (deduplication guard active)

## 0. Setup and imports

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path(".").resolve().parent))

REPO_ROOT   = Path(".").resolve().parent
RESULTS_DIR = REPO_ROOT / "notebooks" / "results"
MODELS_DIR  = REPO_ROOT / "models" / "visual"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

experiments_path = RESULTS_DIR / "experiments.csv"

# Check Hailo platform is available
try:
    from hailo_platform import HEF, VDevice, HailoSchedulingAlgorithm
    HAILO_AVAILABLE = True
    print("hailo_platform: available")
except ImportError:
    HAILO_AVAILABLE = False
    print("WARNING: hailo_platform not available — run this notebook on the Pi")

# Check PyTorch for CPU baseline
try:
    import timm
    import torch
    TORCH_AVAILABLE = True
    print(f"PyTorch: {torch.__version__}")
    print(f"timm:    {timm.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("WARNING: torch/timm not available")

print(f"\nRepo root: {REPO_ROOT}")
print(f"Results dir: {RESULTS_DIR}")

## 1. Helper — experiments.csv deduplication guard

Identical to the guard used in all other Avis notebooks.
Re-running cells never creates duplicate rows.

In [ ]:
def append_if_new(csv_path: Path, row: dict) -> None:
    """
    Append a result row to experiments.csv if the (notebook, model) pair
    is not already present. Safe to call on every kernel restart.
    """
    new_df = pd.DataFrame([row])
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        key_cols = ["notebook", "model"]
        mask = (
            (existing["notebook"] == row["notebook"]) &
            (existing["model"]    == row["model"])
        )
        if mask.any():
            print(f"experiments.csv — '{row['model']}' already recorded, skipping.")
            return
        updated = pd.concat([existing, new_df], ignore_index=True)
    else:
        updated = new_df
    updated.to_csv(csv_path, index=False)
    print(f"experiments.csv — appended: {row['model']}")

## 2. HEF paths and model info

Two HEF files are used in this notebook:

- `resnet_v1_50_h8l.hef` — Pre-compiled by Hailo, ships with the hailo-models package.
  Used for the raw hardware throughput benchmark (no scheduler). Not bird-specific.
- `efficientnet_b0_avis_v2.hef` — Our trained EfficientNet-B0 feature extractor,
  compiled from `frozen_extractor.pt` via DFC 3.32.0 and calibrated on real SD bird images.
  See Section 7 for the full compilation story.

In [ ]:
RESNET_HEF_PATH     = Path("/usr/share/hailo-models/resnet_v1_50_h8l.hef")
EFFICIENTNET_HEF_PATH = MODELS_DIR / "efficientnet_b0_avis_v2.hef"
CPU_EXTRACTOR_PATH  = MODELS_DIR / "frozen_extractor.pt"

print(f"ResNet-50 HEF       : {RESNET_HEF_PATH}  exists={RESNET_HEF_PATH.exists()}")
print(f"EfficientNet HEF    : {EFFICIENTNET_HEF_PATH}  exists={EFFICIENTNET_HEF_PATH.exists()}")
print(f"CPU extractor .pt   : {CPU_EXTRACTOR_PATH}  exists={CPU_EXTRACTOR_PATH.exists()}")

if HAILO_AVAILABLE and EFFICIENTNET_HEF_PATH.exists():
    hef = HEF(str(EFFICIENTNET_HEF_PATH))
    for ng in hef.get_network_group_names():
        inputs  = hef.get_input_vstream_infos(ng)
        outputs = hef.get_output_vstream_infos(ng)
        print(f"\nEfficientNet HEF network group: {ng}")
        print(f"  Inputs:  {[(v.name, v.shape) for v in inputs]}")
        print(f"  Outputs: {[(v.name, v.shape) for v in outputs]}")

## 3. CPU baseline — EfficientNet-B0 inference latency

This is our current production path: PyTorch forward pass through the frozen
EfficientNet-B0 backbone on CPU. The output is a 1280-dim feature vector that
feeds into the sklearn LogReg head.

**Why measure this separately from the full classify() pipeline?**
The neural backbone is the compute bottleneck. LogReg inference is ~0.1ms.
We isolate the backbone to make the Hailo comparison apples-to-apples.

In [ ]:
if not TORCH_AVAILABLE:
    print("Skipping — torch/timm not available")
else:
    N_RUNS = 50

    ckpt  = torch.load(str(CPU_EXTRACTOR_PATH), map_location="cpu")
    model = timm.create_model(
        ckpt["model_name"],
        pretrained=False,
        num_classes=0,
        global_pool=ckpt["global_pool"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    print(f"Loaded EfficientNet-B0 from {CPU_EXTRACTOR_PATH.name}")
    print(f"Model: {ckpt['model_name']}  feature_dim={ckpt['feature_dim']}")

    # Warmup — first pass loads weights into CPU cache
    dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy)

    # Timed runs
    cpu_times = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            dummy = torch.randn(1, 3, 224, 224)
            t0 = time.perf_counter()
            feat = model(dummy)
            cpu_times.append((time.perf_counter() - t0) * 1000)

    cpu_times = np.array(cpu_times)
    print(f"\nCPU EfficientNet-B0 ({N_RUNS} runs):")
    print(f"  mean:   {cpu_times.mean():.2f} ms")
    print(f"  median: {np.median(cpu_times):.2f} ms")
    print(f"  min:    {cpu_times.min():.2f} ms")
    print(f"  std:    {cpu_times.std():.2f} ms")
    print(f"  output shape: {feat.shape}")

## 4. Hailo benchmark — Part A: Raw hardware throughput (ResNet-50)

Measures the maximum inference throughput of the HAILO8L chip using a
pre-compiled ResNet-50 HEF. This run uses the default VDevice (no ROUND_ROBIN
scheduler) to measure raw chip speed without Python scheduling overhead.

**Why ResNet-50?**
It ships pre-compiled for HAILO8L in the hailo-models package. Running it here
establishes an upper bound on hardware capability that any compiled model can approach.

**Key note on dtype:**
Hailo compiled models use INT8 quantization internally. Input and output buffers
must be `uint8`, not `float32`. The chip handles all quantization/dequantization
internally at the hardware layer.

In [ ]:
if not HAILO_AVAILABLE:
    print("Skipping — hailo_platform not available")
    resnet_times = np.array([0.21] * 50)  # recorded result for plotting
    print("Using recorded result: mean=0.21ms")
else:
    N_RUNS = 50
    resnet_times = []

    with VDevice() as target:
        model_hailo = target.create_infer_model(str(RESNET_HEF_PATH))
        model_hailo.set_batch_size(1)
        with model_hailo.configure() as configured_model:

            # Warmup
            for _ in range(5):
                dummy = np.random.randint(0, 256, (1, 224, 224, 3), dtype=np.uint8)
                out   = np.empty((1, 1000), dtype=np.uint8)
                b = configured_model.create_bindings()
                b.input().set_buffer(dummy)
                b.output().set_buffer(out)
                configured_model.run([b], timeout=1000)

            # Timed runs
            for _ in range(N_RUNS):
                dummy = np.random.randint(0, 256, (1, 224, 224, 3), dtype=np.uint8)
                out   = np.empty((1, 1000), dtype=np.uint8)
                b = configured_model.create_bindings()
                b.input().set_buffer(dummy)
                b.output().set_buffer(out)
                t0 = time.perf_counter()
                configured_model.run([b], timeout=1000)
                resnet_times.append((time.perf_counter() - t0) * 1000)

    resnet_times = np.array(resnet_times)
    print(f"Hailo ResNet-50 — raw throughput ({N_RUNS} runs):")
    print(f"  mean:   {resnet_times.mean():.2f} ms")
    print(f"  median: {np.median(resnet_times):.2f} ms")
    print(f"  min:    {resnet_times.min():.2f} ms")
    print(f"  std:    {resnet_times.std():.2f} ms")

## 5. Hailo benchmark — Part B: Production EfficientNet-B0 (ROUND_ROBIN)

Benchmarks our actual trained model — `efficientnet_b0_avis_v2.hef` — under
the same conditions used in production.

**Why ROUND_ROBIN scheduling?**
HailoRT 4.23.0 requires `HailoSchedulingAlgorithm.ROUND_ROBIN` to produce
correct non-zero output from the `InferModel` API. Without it, every inference
returns `HAILO_STREAM_NOT_ACTIVATED(72)` and fills the output buffer with zeros.
This is a HailoRT 4.23.0 API requirement, not a hardware limitation.

**What the latency includes:**
- Python API call overhead (~1ms)
- ROUND_ROBIN scheduler dispatch latency (~12ms)
- Actual NPU inference (<1ms)

The scheduler overhead dominates at batch_size=1. For our bird detection loop
(one inference per camera frame every 1–5 seconds), 13ms is more than acceptable.

In [ ]:
if not HAILO_AVAILABLE or not EFFICIENTNET_HEF_PATH.exists():
    print("Skipping — hailo_platform or HEF not available")
    hailo_eff_times = np.array([13.13] * 50)  # recorded result for plotting
    print("Using recorded result: mean=13.13ms")
else:
    N_RUNS = 50
    hailo_eff_times = []

    params = VDevice.create_params()
    params.scheduling_algorithm = HailoSchedulingAlgorithm.ROUND_ROBIN

    with VDevice(params) as target:
        model_hailo = target.create_infer_model(str(EFFICIENTNET_HEF_PATH))
        model_hailo.set_batch_size(1)
        with model_hailo.configure() as configured_model:

            # Pre-allocate buffers once — reuse across runs
            bindings = configured_model.create_bindings()
            out_buf  = np.empty((1, 1280), dtype=np.uint8)
            bindings.output().set_buffer(out_buf)

            # Warmup — scheduler needs a few cycles to stabilise
            for _ in range(5):
                dummy = np.random.randint(0, 256, (1, 224, 224, 3), dtype=np.uint8)
                bindings.input().set_buffer(dummy)
                configured_model.run([bindings], timeout=1000)

            # Timed runs
            for _ in range(N_RUNS):
                dummy = np.random.randint(0, 256, (1, 224, 224, 3), dtype=np.uint8)
                bindings.input().set_buffer(dummy)
                t0 = time.perf_counter()
                configured_model.run([bindings], timeout=1000)
                hailo_eff_times.append((time.perf_counter() - t0) * 1000)

    hailo_eff_times = np.array(hailo_eff_times)
    print(f"Hailo EfficientNet-B0 ROUND_ROBIN ({N_RUNS} runs):")
    print(f"  mean:   {hailo_eff_times.mean():.2f} ms")
    print(f"  median: {np.median(hailo_eff_times):.2f} ms")
    print(f"  min:    {hailo_eff_times.min():.2f} ms")
    print(f"  std:    {hailo_eff_times.std():.2f} ms")
    print(f"\nSpeedup vs CPU: {cpu_times.mean() / hailo_eff_times.mean():.1f}×")

## 6. Results — comparison table and charts

Summarise all three paths in a single comparison table and produce two charts
matching the visual style of the other Avis experiment notebooks.

In [ ]:
# Use recorded values if live run wasn't possible
if 'cpu_times' not in dir():
    cpu_times = np.array([85.43] * 50)
if 'resnet_times' not in dir():
    resnet_times = np.array([0.21] * 50)
if 'hailo_eff_times' not in dir():
    hailo_eff_times = np.array([13.13] * 50)

results = {
    "CPU — EfficientNet-B0\n(PyTorch, our model)": cpu_times,
    "Hailo — ResNet-50\n(raw throughput, no scheduler)": resnet_times,
    "Hailo — EfficientNet-B0\n(our model, ROUND_ROBIN)": hailo_eff_times,
}

print(f"{'Path':<45} {'Mean (ms)':>10} {'Median (ms)':>12} {'Min (ms)':>10} {'Std':>8}")
print("-" * 90)
for label, times in results.items():
    label_flat = label.replace("\n", " ")
    print(f"{label_flat:<45} {times.mean():>10.2f} {np.median(times):>12.2f} {times.min():>10.2f} {times.std():>8.2f}")

speedup_raw  = cpu_times.mean() / resnet_times.mean()
speedup_prod = cpu_times.mean() / hailo_eff_times.mean()
print(f"\nRaw hardware speedup  (ResNet-50):       {speedup_raw:.0f}×")
print(f"Production speedup    (EfficientNet-B0): {speedup_prod:.1f}×")

In [ ]:
# ── Chart 1: Bar chart comparison ────────────────────────────────────────────
labels  = list(results.keys())
means   = [t.mean() for t in results.values()]
stds    = [t.std()  for t in results.values()]
colors  = ["#4c72b0", "#55a868", "#c44e52"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, alpha=0.85, edgecolor="white", linewidth=1.2)

# Annotate bars with mean values
for bar, mean, std in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.8,
        f"{mean:.2f} ms",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax.set_ylabel("Inference latency (ms)", fontsize=12)
ax.set_title("EfficientNet-B0 Inference Latency: CPU vs Hailo HAILO8L", fontsize=13, pad=15)
ax.set_ylim(0, max(means) * 1.25)
ax.tick_params(axis="x", labelsize=9)
ax.spines[["top", "right"]].set_visible(False)

# Speedup annotations
ax.annotate(
    f"{speedup_raw:.0f}× faster\nthan CPU",
    xy=(1, means[1]), xytext=(1, means[1] + max(means) * 0.15),
    ha="center", fontsize=9, color="#55a868",
    arrowprops=dict(arrowstyle="->", color="#55a868"),
)
ax.annotate(
    f"{speedup_prod:.1f}× faster\nthan CPU",
    xy=(2, means[2]), xytext=(2, means[2] + max(means) * 0.15),
    ha="center", fontsize=9, color="#c44e52",
    arrowprops=dict(arrowstyle="->", color="#c44e52"),
)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "hailo_benchmark_comparison.png", dpi=150)
plt.show()
print("Saved: hailo_benchmark_comparison.png")

In [ ]:
# ── Chart 2: Latency distribution (box plot) ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

flat_labels = [l.replace("\n", " ") for l in labels]
bp = ax.boxplot(
    list(results.values()),
    labels=flat_labels,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
)

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("Inference latency (ms)", fontsize=12)
ax.set_title("Latency Distribution — 50 runs per path", fontsize=13)
ax.tick_params(axis="x", labelsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "hailo_latency_distribution.png", dpi=150)
plt.show()
print("Saved: hailo_latency_distribution.png")

## 7. Compilation pipeline — from `.pt` to `.hef`

This section documents every step taken to compile our trained EfficientNet-B0
feature extractor to a Hailo `.hef` binary. All decisions are recorded here
so the result is reproducible. The full automated script lives at
`scripts/compile_hailo_hef.py`.

### Why the pipeline is non-trivial

Hailo's Dataflow Compiler (DFC) converts ONNX models to `.hef` files targeting
a specific HailoRT runtime version. The Pi ships with HailoRT 4.23.0, but the
publicly available DFC for Hailo-8/8L is version 3.32.0 (AI SW Suite 2025-07),
which targets HailoRT 4.22.0. **Forward compatibility was confirmed** — a `.hef`
compiled for 4.22.0 loads and runs correctly on 4.23.0.

### Steps

| Step | Tool | Input | Output | Key decision |
|------|------|-------|--------|--------------|
| 1 | PyTorch `torch.onnx.export` | `frozen_extractor.pt` | `efficientnet_b0_avis_features.onnx` (15.3MB) | `num_classes=0, global_pool='avg'` — backbone only, no classification head |
| 2 | `hailo parser onnx` (DFC 3.32.0) | `.onnx` | `efficientnet_b0_avis.har` | `--end-node-names /global_pool/pool/GlobalAveragePool` — stop before Flatten (unsupported op) |
| 3 | `hailo_sdk_client.ClientRunner.optimize` | `.har` + calibration images | `efficientnet_b0_avis_quantized.har` | `pre_quantization_optimization(global_avgpool_reduction, ...)` to fix SE block shift delta error; 64 real SD bird images as calibration |
| 4 | `hailo compiler` (DFC 3.32.0) | quantized `.har` | `efficientnet_b0_avis_v2.hef` (13MB) | 6 hardware contexts, `hw-arch hailo8l` |

### Key technical issue resolved: SE block avgpool shift delta

EfficientNet-B0's Squeeze-and-Excitation blocks include a global average pool
that collapses a 7×7 spatial feature map to 1×1. The ratio between input and
output spatial dimensions creates a shift delta of 3.53 in the quantizer's
fixed-point arithmetic — above the HAILO8L hardware limit of 2.0.

**Fix:** A model script command inserted before quantization that pre-reduces
the avgpool dimensions, keeping the shift delta within hardware limits:

```
pre_quantization_optimization(global_avgpool_reduction, layers=efficientnet_b0_avis/avgpool1, division_factors=[7, 7])
```

### Compilation environment

- **DFC:** Hailo AI SW Suite 2025-07 Docker (`hailo8_ai_sw_suite_2025-07_docker.zip`)
- **DFC version:** 3.32.0 (targets HailoRT 4.22.0)
- **Host:** WSL2 Ubuntu 24.04 on Windows, Docker 29.4.0
- **Pi runtime:** HailoRT 4.23.0 (forward-compatible confirmed)
- **Calibration data:** 64 real SD bird images from `data/splits/visual_train.csv`,
  resized to 224×224 uint8 then normalized to float32 [0, 1] for quantizer

In [ ]:
# ── Reproduce Step 1: ONNX export ────────────────────────────────────────────
# Run on the laptop (Windows or Linux) — not required on Pi
# Requires: pip install onnx

ONNX_EXPORT_CODE = '''
import torch
import timm

ckpt  = torch.load("models/visual/frozen_extractor.pt", map_location="cpu")
model = timm.create_model(
    ckpt["model_name"], pretrained=False,
    num_classes=0, global_pool=ckpt["global_pool"]
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy,
    "efficientnet_b0_avis_features.onnx",
    input_names=["input"],
    output_names=["features"],
    opset_version=11,
)
# Output: efficientnet_b0_avis_features.onnx  (15.3 MB)
# Verified: output shape torch.Size([1, 1280])
'''
print("Step 1 — ONNX export (run on laptop):")
print(ONNX_EXPORT_CODE)

In [ ]:
# ── Reproduce Steps 2–4: DFC compilation (run inside Hailo Docker) ────────────
# Docker image: hailo8_ai_sw_suite_2025-07.tar.gz from Hailo developer zone
# Load: sudo docker load -i hailo8_ai_sw_suite_2025-07.tar.gz
# Run:  sudo docker run -it --rm \
#         -v ~/hailo_suite/shared_with_docker:/workspace \
#         hailo8_ai_sw_suite_2025-07:1 bash

DFC_COMPILE_CODE = '''
# Inside Docker container — DFC 3.32.0

# Step 2: Parse ONNX to HAR
# --end-node-names stops before Flatten layer (unsupported by Hailo parser)
hailo parser onnx /workspace/efficientnet_b0_avis_features.onnx \\
  --hw-arch hailo8l \\
  --net-name efficientnet_b0_avis \\
  --har-path /tmp/efficientnet_b0_avis.har \\
  --end-node-names /global_pool/pool/GlobalAveragePool \\
  -y
# Output: /tmp/efficientnet_b0_avis.har

# Step 3: Quantize with real calibration data
# Model script fixes SE block avgpool shift delta (3.53 > 2.0 hardware limit)
# Python script — see scripts/compile_hailo_hef.py for full version
python3 << \'EOF\'
import numpy as np
from hailo_sdk_client import ClientRunner

runner = ClientRunner(har="/tmp/efficientnet_b0_avis.har")

# SE block fix: pre-reduce avgpool spatial dimensions to keep shift delta < 2.0
with open("/tmp/avis_se_fix.alls", "w") as f:
    f.write("pre_quantization_optimization(global_avgpool_reduction, "
            "layers=efficientnet_b0_avis/avgpool1, division_factors=[7, 7])\\n")

runner.load_model_script("/tmp/avis_se_fix.alls")

# 64 real SD bird images, uint8 resized to 224x224, normalized to float32 [0,1]
calib = np.load("/workspace/calib_images.npy").astype(np.float32) / 255.0
runner.optimize(calib)
runner.save_har("/tmp/efficientnet_b0_avis_quantized.har")
EOF

# Step 4: Compile to HEF
hailo compiler /tmp/efficientnet_b0_avis_quantized.har \\
  --hw-arch hailo8l \\
  --output-dir /tmp/
# Output: /tmp/efficientnet_b0_avis.hef  (13 MB)
# Copy to Pi: scp /tmp/efficientnet_b0_avis.hef birdfeeder01@<pi-ip>:models/visual/efficientnet_b0_avis_v2.hef
'''
print("Steps 2–4 — DFC compilation (run inside Hailo Docker):")
print(DFC_COMPILE_CODE)

## 8. Append results to experiments.csv

In [ ]:
# Use actual measured values or recorded fallbacks
cpu_mean    = float(cpu_times.mean())         if 'cpu_times'        in dir() else 85.43
resnet_mean = float(resnet_times.mean())      if 'resnet_times'     in dir() else 0.21
hailo_mean  = float(hailo_eff_times.mean())   if 'hailo_eff_times'  in dir() else 13.13

# Row 1: CPU baseline
append_if_new(experiments_path, {
    "phase":       6,
    "notebook":    "hailo_benchmark.ipynb",
    "modality":    "visual",
    "model":       "CPU EfficientNet-B0 inference baseline",
    "n_species":   19,
    "n_test":      50,
    "accuracy":    None,
    "macro_f1":    None,
    "weighted_f1": None,
    "notes":       f"PyTorch CPU forward pass, mean={cpu_mean:.2f}ms. Backbone only (no LogReg head). Reference for Hailo comparison.",
})

# Row 2: Hailo raw throughput
append_if_new(experiments_path, {
    "phase":       6,
    "notebook":    "hailo_benchmark.ipynb",
    "modality":    "visual",
    "model":       "Hailo HAILO8L ResNet-50 raw throughput",
    "n_species":   1000,
    "n_test":      50,
    "accuracy":    None,
    "macro_f1":    None,
    "weighted_f1": None,
    "notes":       f"Pre-compiled ResNet-50 HEF, default VDevice (no ROUND_ROBIN). mean={resnet_mean:.2f}ms. Establishes raw NPU throughput upper bound.",
})

# Row 3: Hailo EfficientNet production
append_if_new(experiments_path, {
    "phase":       6,
    "notebook":    "hailo_benchmark.ipynb",
    "modality":    "visual",
    "model":       "Hailo HAILO8L EfficientNet-B0 (ours, ROUND_ROBIN)",
    "n_species":   19,
    "n_test":      50,
    "accuracy":    None,
    "macro_f1":    None,
    "weighted_f1": None,
    "notes":       f"Our trained frozen_extractor.pt compiled via DFC 3.32.0 to HEF. ROUND_ROBIN scheduler required. mean={hailo_mean:.2f}ms = {cpu_mean/hailo_mean:.1f}x speedup vs CPU. Calibrated on 64 real SD bird images.",
})

## 9. Summary

| Path | Model | Latency | Speedup vs CPU |
|------|-------|---------|----------------|
| CPU PyTorch | EfficientNet-B0 (ours) | 85.43 ms | — |
| Hailo HAILO8L (raw) | ResNet-50 (pre-compiled) | 0.21 ms | ~407× |
| Hailo HAILO8L (prod) | EfficientNet-B0 (ours) | 13.13 ms | **6.5×** |

### Interpretation

The 407× raw throughput figure represents the chip's maximum capability at
saturation — what it can do when no Python scheduler overhead is involved.
The 6.5× production figure represents the real wall-clock improvement for
our single-inference-per-frame use case under the ROUND_ROBIN scheduler.

For the Avis bird detection loop — which runs one inference every 1–5 seconds
depending on motion activity — 13ms vs 85ms is a meaningful improvement:
it frees the Pi 5 CPU for audio processing, image logging, and the agent loop
while the Hailo NPU handles the most compute-intensive task independently.

### Future work

- **Per-class accuracy on HEF model:** Verify that quantization has not degraded
  species-level accuracy vs the float32 model (F1=0.931). Run the held-out test
  set through the Hailo inference path and compare per-class F1.
- **Proper calibration:** Re-quantize with the full 2003-image training set
  (vs 64 images used here) to maximise quantization accuracy.
- **YOLOv8 bird detection:** `yolov8s_h8l.hef` is already on the Pi at
  `/usr/share/hailo-models/`. A YOLO detect-then-classify pipeline would
  replace the fixed `feeder_crop` ROI with per-bird bounding boxes.
- **HailoRT upgrade:** HailoRT 4.23.0 has no matching public DFC. Upgrading
  to a version covered by the AI SW Suite would enable recompilation with
  GPU-accelerated optimization (higher quantization accuracy, lower latency).